In [3]:
print('Hello World')

Hello World


In [4]:

from typing import List, TypedDict, Literal
from pydantic import BaseModel, Field
import time

from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate

from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv

load_dotenv()

True

In [5]:
from langchain_pinecone import PineconeVectorStore
from pinecone import Pinecone, ServerlessSpec

c:\Users\TPWODL\miniconda3\envs\genai\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 1. Data Ingestion Layer (Raw Data Entry)

In [6]:
docs = PyPDFLoader(r"C:\Users\TPWODL\New folder_Content\Advanced_RAG_Project\Supply Code, 2019 with Gazette.pdf").load()

In [7]:
docs[3]

Document(metadata={'producer': 'GPL Ghostscript 9.06', 'creator': 'PScript5.dll Version 5.2', 'creationdate': '2019-10-19T13:51:32+05:30', 'author': 'spm', 'moddate': '2019-11-06T13:21:17+05:30', 'title': 'Microsoft Word - Supply Code, 2019 Final 19.10.2019', 'source': 'C:\\Users\\TPWODL\\New folder_Content\\Advanced_RAG_Project\\Supply Code, 2019 with Gazette.pdf', 'total_pages': 95, 'page': 3, 'page_label': '4'}, page_content='4 \n \n(9) “Authorised Representative ” of any person/entity means all officers, staff, \nrepresentatives or persons discharging functions un der the general or specific \nauthority of the concerned person/entity; \n(10) “Average Power Factor”  means the power factor resulting from variations o f the \nquantum and duration of the consumer’s load during a specific period and its \nvalue corrected to the nearest percentum figure to be calculated as a ratio of the \nregistration during the same period of kilowatt-hour and kilovolt-ampere hour; \n(11) “Bi-direction

In [8]:
print(type(docs))

<class 'list'>


# 2. Data Preprocessing & Cleaning Layer

In [9]:
docs[0]


Document(metadata={'producer': 'GPL Ghostscript 9.06', 'creator': 'PScript5.dll Version 5.2', 'creationdate': '2019-10-19T13:51:32+05:30', 'author': 'spm', 'moddate': '2019-11-06T13:21:17+05:30', 'title': 'Microsoft Word - Supply Code, 2019 Final 19.10.2019', 'source': 'C:\\Users\\TPWODL\\New folder_Content\\Advanced_RAG_Project\\Supply Code, 2019 with Gazette.pdf', 'total_pages': 95, 'page': 0, 'page_label': '1'}, page_content='1 \n \n \n \n \n \n \n \n \n \n \n \n \n \n \nODISHA ELECTRICITY REGULATORY COMMISSION \nBHUBANESWAR – 751 021 \n***** \n \nNOTIFICATION \nThe 27 th  August, 2019 \n \nNo.OERC/Engg/92/2003(Vol-VII)/1210 - In exercise of  the powers conferred by Section \n50 read with and Section 181 (2) (t), (v), (w) and (x) read with Part-VI of the Electricity Act, \n2003 (Act 36 of 2003) and all other powers enabling  it in this behalf, the Odisha Electricity \nRegulatory Commission hereby make the following Sup ply Code by Notification to govern \nsupply of electricity by th

In [10]:
len(docs)

95

# 3. Chunking & Document Structuring Layer

In [11]:

chunks = RecursiveCharacterTextSplitter(
    chunk_size=200, chunk_overlap=30
).split_documents(docs)

In [12]:
len(chunks)

1545

In [13]:
print(type(chunks))

<class 'list'>


In [14]:
chunks[0]

Document(metadata={'producer': 'GPL Ghostscript 9.06', 'creator': 'PScript5.dll Version 5.2', 'creationdate': '2019-10-19T13:51:32+05:30', 'author': 'spm', 'moddate': '2019-11-06T13:21:17+05:30', 'title': 'Microsoft Word - Supply Code, 2019 Final 19.10.2019', 'source': 'C:\\Users\\TPWODL\\New folder_Content\\Advanced_RAG_Project\\Supply Code, 2019 with Gazette.pdf', 'total_pages': 95, 'page': 0, 'page_label': '1'}, page_content='1 \n \n \n \n \n \n \n \n \n \n \n \n \n \n \nODISHA ELECTRICITY REGULATORY COMMISSION \nBHUBANESWAR – 751 021 \n***** \n \nNOTIFICATION \nThe 27 th  August, 2019')

In [15]:
import os
from dotenv import load_dotenv
from pinecone import Pinecone, ServerlessSpec  # Added ServerlessSpec
from langchain_pinecone import PineconeVectorStore # Corrected import
from langchain_openai import OpenAIEmbeddings

In [16]:
# 1. Load the .env file (Must be at the top)
load_dotenv()

True

# 4. Embedding Generation Layer

# 5. Vector Storage & Indexing Layer

In [17]:
# 2. Initialize Embeddings 
# Note: Ensure OPENAI_API_KEY is in your .env file
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 3. Initialize Pinecone client
api_key = os.getenv("PINECONE_API_KEY")
if not api_key:
    raise ValueError("PINECONE_API_KEY not found. Check your .env file!")

pc = Pinecone(api_key=api_key)
index_name = "rag-index-project"

# 4. FIX: Delete the old index if it has the wrong dimensions (512)
# You only need to do this ONCE. 
if index_name in pc.list_indexes().names():
    idx_description = pc.describe_index(index_name)
    if idx_description.dimension != 1536:
        print(f"Deleting index {index_name} because dimensions don't match...")
        pc.delete_index(index_name)

# 5. Create index with correct 1536 dimensions
if index_name not in pc.list_indexes().names():
    print(f"Creating new index: {index_name}")
    pc.create_index(
        name=index_name,
        dimension=1536, 
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )

# 6. Connect to index and store documents
vector_store = PineconeVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings,
    index_name=index_name
)

# 7. Create retriever
retriever = vector_store.as_retriever(search_kwargs={"k": 4})
print("Success! Documents are stored and retriever is ready.")


Success! Documents are stored and retriever is ready.


# 6. Retrieval Layer (Core of RAG)

In [18]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [20]:
query = "What is Application for power Supply?"

In [42]:
retriv_docs = retriever.invoke(query)

In [43]:
retriv_docs

[Document(id='65a52ce2-5313-4ea6-a052-8101e0c222d3', metadata={'author': 'spm', 'creationdate': '2019-10-19T13:51:32+05:30', 'creator': 'PScript5.dll Version 5.2', 'moddate': '2019-11-06T13:21:17+05:30', 'page': 10.0, 'page_label': '11', 'producer': 'GPL Ghostscript 9.06', 'source': 'C:\\Users\\TPWODL\\New folder_Content\\Advanced_RAG_Project\\Supply Code, 2019 with Gazette.pdf', 'title': 'Microsoft Word - Supply Code, 2019 Final 19.10.2019', 'total_pages': 95.0}, page_content='11  \n \nCHAPTER III  \nPOWER SUPPLY \nApplication for Supply \n3. Application for initial supply or subsequent additi onal supply of power shall be made in'),
 Document(id='4e303115-6166-407c-8b55-f0e51492e44c', metadata={'author': 'spm', 'creationdate': '2019-10-19T13:51:32+05:30', 'creator': 'PScript5.dll Version 5.2', 'moddate': '2019-11-06T13:21:17+05:30', 'page': 10.0, 'page_label': '11', 'producer': 'GPL Ghostscript 9.06', 'source': 'C:\\Users\\TPWODL\\New folder_Content\\Advanced_RAG_Project\\Supply Code

In [44]:
print(type(retriv_docs))

<class 'list'>


In [45]:
retriv_docs[0]

Document(id='65a52ce2-5313-4ea6-a052-8101e0c222d3', metadata={'author': 'spm', 'creationdate': '2019-10-19T13:51:32+05:30', 'creator': 'PScript5.dll Version 5.2', 'moddate': '2019-11-06T13:21:17+05:30', 'page': 10.0, 'page_label': '11', 'producer': 'GPL Ghostscript 9.06', 'source': 'C:\\Users\\TPWODL\\New folder_Content\\Advanced_RAG_Project\\Supply Code, 2019 with Gazette.pdf', 'title': 'Microsoft Word - Supply Code, 2019 Final 19.10.2019', 'total_pages': 95.0}, page_content='11  \n \nCHAPTER III  \nPOWER SUPPLY \nApplication for Supply \n3. Application for initial supply or subsequent additi onal supply of power shall be made in')

In [46]:
len(retriv_docs)

4

In [47]:
for i, doc in enumerate(retriv_docs):
    print(f"\nResult {i+1}:")
    print(doc.page_content)
    


Result 1:
11  
 
CHAPTER III  
POWER SUPPLY 
Application for Supply 
3. Application for initial supply or subsequent additi onal supply of power shall be made in

Result 2:
11  
 
CHAPTER III  
POWER SUPPLY 
Application for Supply 
3. Application for initial supply or subsequent additi onal supply of power shall be made in

Result 3:
power supply to the applicant within the period approved by the Commission. 
Provided that if the substation is meant to extend supply to an individual

Result 4:
power supply to the applicant within the period approved by the Commission. 
Provided that if the substation is meant to extend supply to an individual


# 7. Generation Layer (LLM)

In [49]:
from langchain_classic.chains import RetrievalQA

In [50]:
# Create the QA chain
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever
)

# Get the final answer
answer = qa_chain.invoke(query)
print(answer["result"])

An application for power supply is a formal request made by an individual or entity to a utility company or regulatory body for the initial supply of electricity or for additional power supply. This application typically includes details about the applicant, the location where power is needed, and the specific requirements for the supply. The application process is governed by regulations set by the relevant commission or authority, which also approves the timeline for providing the power supply.


In [53]:
query2 = "Which category covers power supply for street lighting and traffic signaling provided by a local authority??"

In [54]:
answer = qa_chain.invoke(query2)
print(answer["result"])

The category that covers power supply for street lighting and traffic signaling provided by a local authority is Public Lighting.


In [ ]:
print(answer["result"])

In [55]:
query3 = "what is data science?"

In [58]:
answer = qa_chain.invoke(query3)
print(answer['result'])

I don't know.


# Answer the query+retriever+llm

In [59]:
query4 = "what is Allied Agricultural Activities?"

In [60]:
answer = qa_chain.invoke(query4)
print(answer['result'])

Allied Agricultural Activities refer to activities that support or complement primary agricultural production. In the context provided, it specifically relates to the supply of power for aquaculture, which includes pisciculture (the breeding and rearing of fish). These activities may involve various processes and operations that enhance agricultural productivity and sustainability.
